# Лабораторная работа №0. Как работает наш сетевой notebook

Это вводная работа. Здесь не нужно знать маршрутизацию, ARP или TCP: цель — освоить один и тот же рабочий цикл, который будет повторяться во всём курсе.

Работаем в двух местах: в notebook и в терминалах `pc1` / `pc2`. Notebook нужен для вопросов, capture и наблюдений; терминалы — для команд на реальных Linux-узлах.

В этой лабораторной CHECK проверяет простой marker-файл. Это специально безопасное исключение для onboarding. Начиная со следующих работ CHECK проверяет именно сетевое состояние: адреса, маршруты, соседей и end-to-end связность.

## Один и тот же цикл во всём курсе

![Цикл лабораторной](assets/workflow.svg)

Сначала вы формулируете ожидание, затем действуете на реальном стенде, наблюдаете результат, объясняете его и только после этого используете CHECK. Потом мы намеренно ломаем одно условие и локализуем причину.

### Q1. Где выполнится Python-код из следующей ячейки: внутри `pc1` или в отдельном Jupyter kernel? Что вы ожидаете увидеть в `hostname`?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду, готовый вывод эксперимента или готовое решение.  

</details>

In [ ]:
import socket
print("hostname Jupyter kernel:", socket.gethostname())
print("Python выполняется в kernel notebook, а не внутри pc1.")

### Наблюдение

Имя, которое вывела Python-cell, не обязано быть `pc1`. Это отдельное окружение notebook.

Команды, которые должны менять или исследовать сетевой узел, мы выполняем в его terminal. Поэтому результат Python-cell сам по себе ничего не доказывает о состоянии интерфейсов внутри `pc1`.

## 1. Откройте реальный terminal

Откройте terminal `pc1` и выполните:

```bash
hostname
ip -br link
ip -br addr
```

Затем откройте `pc2` и выполните только:

```bash
hostname
ip -br addr
```

Для onboarding учебный линк `eth1` уже настроен: `pc1` имеет `192.0.2.10/24`, `pc2` — `192.0.2.20/24`. `eth0` — management plane платформы; его не меняем.

## Схема стенда

![Минимальный стенд](assets/topology.svg)

### Q2. Если во время capture на `pc1:eth1` вы запустите `ping 192.0.2.20`, останется ли таблица захвата пустой? Почему?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду, готовый вывод эксперимента или готовое решение.  

</details>

## 2. Захватим настоящий трафик

Запустите следующую cell. Пока она работает, сразу перейдите в terminal `pc1` и выполните:

```bash
ping -c 3 192.0.2.20
```

Мы пока не разбираем поля пакета. Важно увидеть сам workflow: действие в terminal → реальные кадры на интерфейсе → интерактивный viewer в notebook.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp" --timeout 10 --packets 30 --save captures/onboarding-icmp.pcap
import time
print("Сейчас выполните в terminal pc1: ping -c 3 192.0.2.20")
time.sleep(8)

### Наблюдение

Viewer показывает пакеты, которые реально прошли через `pc1:eth1`. Это не нарисованная симуляция.

Ключевые capture в курсе сохраняются как `.pcap`, а важный результат дублируется обычным output notebook. Поэтому преподаватель сможет увидеть его и в статическом HTML/PDF.

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

viewer = PcapViewer("captures/onboarding-icmp.pcap", "icmp", limit=20)
frames = viewer.packets()
display(frames)

icmp = frames[
    frames["Protocol"].astype(str).str.contains("ICMP", case=False, na=False)
]
if not icmp.empty:
    frame_number = int(icmp.iloc[0]["№"])
    tree = viewer.packet(frame_number, "protocols")
    print(f"\nProtocol tree · frame {frame_number}\n")
    print("\n".join(tree.splitlines()[:60]))
else:
    print("ICMP-кадр не найден: повторите capture и ping.")

## 3. Первый успешный CHECK

Сейчас познакомимся с CHECK без сетевой задачи. В **Lab 000 только** checker смотрит на marker `/tmp/cms-lab-ready` внутри `pc1`.

В terminal `pc1`:

```bash
printf 'ready\n' > /tmp/cms-lab-ready
cat /tmp/cms-lab-ready
```

Запустите **CHECK**.

Ожидаемый результат — зелёная проверка. Важно: checker не запоминает, какую команду вы вводили. Он как внешний оператор подключается к стенду и смотрит на наблюдаемое состояние.

### Q3. Что произойдёт с CHECK, если удалить marker-файл, не меняя ничего больше? Какой результат вы ожидаете увидеть?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q3, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду, готовый вывод эксперимента или готовое решение.  

</details>

## 4. Намеренно сломаем одно условие

На `pc1`:

```bash
rm -f /tmp/cms-lab-ready
```

Снова запустите **CHECK**.

Не исправляйте состояние сразу. Прочитайте текст задачи и log. Здесь важно отличить два случая:

- CHECK запустился, но условие лабораторной не выполнено;
- checker вообще не смог выполниться из-за инфраструктурной ошибки.

В первом случае вы получаете диагностический результат про состояние стенда — именно это сейчас и должно произойти.

## 5. Диагностика: локализуйте причину

Не угадывайте и не повторяйте команды вслепую. Сначала подтвердите наблюдение:

```bash
ls -l /tmp/cms-lab-ready
cat /tmp/cms-lab-ready
```

Если в интерфейсе доступен **HINT**, можно открыть его сейчас. Хорошая подсказка должна направить к следующему диагностическому действию, а не выполнить исправление за вас.

Причина локализована: checker ожидал конкретное состояние внутри `pc1`, а marker отсутствует.

### Q4. Что в этой работе проверяет CHECK: факт выполнения конкретной команды или итоговое наблюдаемое состояние `pc1`? Почему это различие важно для следующих лабораторных?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4, который находится выше.  
Не отвечай на вопрос сам и не пиши эталонный ответ.  
Скажи, есть ли в моём рассуждении логическая ошибка или пропущенный шаг, и задай 1–2 наводящих вопроса.  
Не давай готовую команду, готовый вывод эксперимента или готовое решение.  

</details>

## 6. Исправление другим способом

Верните то же ожидаемое состояние, но **другой командой**, чем перед первым CHECK:

```bash
echo ready > /tmp/cms-lab-ready
cat /tmp/cms-lab-ready
```

Запустите **CHECK** ещё раз. Он должен снова пройти.

Получился полный цикл:

```text
predict → act → observe → explain → CHECK → break → diagnose → fix
```

### Наблюдение и объяснение

Первый раз marker создавался через `printf`, после поломки — через `echo`. CHECK прошёл в обоих случаях, когда итоговое содержимое было `ready`.

Значит, checker наблюдает состояние стенда, а не историю введённых команд. В следующих лабораторных тот же принцип применяется к сетевым свойствам.

## 7. Что делают HINT и RESET

**HINT** — это следующий диагностический ориентир, когда вы застряли.

**RESET** пересоздаёт попытку и возвращает topology к исходному состоянию. Если выполнить RESET после финального зелёного CHECK, marker исчезнет, потому что он не входит в исходную конфигурацию. Тогда для повторного финального CHECK нужно снова создать marker.

Такой же принцип останется дальше: reset возвращает стенд, а не «ответы в Markdown».

## Что нужно унести из Lab 000

- notebook kernel и terminal сетевого узла — разные окружения;
- команды на `pc1` / `pc2` работают с настоящими Linux network namespaces;
- `%%capture_traffic` показывает реальный трафик интерфейса и сохраняет PCAP;
- важные capture имеют статический notebook-output для HTML/PDF;
- CHECK смотрит на наблюдаемое состояние, а не на историю введённых команд;
- failed CHECK — это часть диагностики, если checker сам запустился;
- основной цикл курса: **predict → act → observe → explain → CHECK → break → diagnose → fix**.

В Lab 001 этот же UX уже будет применяться не к marker, а к одному реальному пакету через стек.